# MobileBERT (google/mobilebert-uncased) — Tagalog NSFW Text Classifier

Trains MobileBERT with fixed, already-tuned hyperparameters and exports ONNX + TFLite (litert-torch float32/INT8). For the Optuna search that produced these hyperparameters, see `MobileBERT_hyperparameters.ipynb`.

In [1]:
!pip install onnxscript optimum[onnxruntime] onnxconverter-common litert-torch torchao

INFO: pip is looking at multiple versions of onnx to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of optimum-onnx[onnxruntime] to determine which version is compatible with other requirements. This could take a while.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 722.0/722.0 kB 31.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.2/161.2 kB 20.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.5/89.5 kB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 575.8/575.8 kB 54.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.3/419.3 kB 40.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.6/17.6 MB 125.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.6/117.6 MB 22.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 M

In [2]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import (
    MobileBertTokenizer,
    MobileBertForSequenceClassification,
    get_linear_schedule_with_warmup
)
from torch.optim import AdamW
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    average_precision_score
)
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import warnings
import time
import os
import shutil
import json
from dataclasses import dataclass, field
from typing import Tuple, List, Dict, Any, Optional
import logging
import sys

warnings.filterwarnings('ignore')

In [3]:
@dataclass
class Config:
    seed: int = 42
    verbose: bool = True
    model_name: str = 'google/mobilebert-uncased'
    csv_path: str = "cleaned_data.csv"
    text_col: str = "text"
    label_col: str = "label"
    max_length: int = 128
    batch_size: int = 32  # HPO-tuned
    learning_rate: float = 2.625530530803658e-05  # HPO-tuned (see MobileBERT_hyperparameters.ipynb)
    weight_decay: float = 0.22092145073929378  # HPO-tuned
    dropout: float = 0.16600674518398445  # HPO-tuned
    warmup_ratio: float = 0.12415965365695053  # HPO-tuned
    epochs: int = 50
    device: str = 'cuda' if torch.cuda.is_available() else 'cpu'
    output_dir: str = "."
    label_mapping: Dict[str, int] = field(default_factory=lambda: {'safe': 0, 'nsfw': 1})
    gradient_accumulation_steps: int = 1
    patience: int = 3

    # Dataset Split Ratios
    train_ratio: float = 0.70
    val_ratio: float = 0.25
    test_ratio: float = 0.05

    # Output Paths
    model_save_name: str = 'best_mobilebert_nsfw_model.pth'
    tokenizer_save_dir: str = 'best_mobilebert_tokenizer'
    onnx_path: str = "mobilebert_nsfw.onnx"
    tflite_path: str = "mobilebert_nsfw.tflite"

    # Visualization Paths
    plot_cm_path: str = 'confusion_matrix_mobilebert.png'
    plot_history_path: str = 'training_history_mobilebert.png'
    plot_metrics_path: str = 'metrics_comparison_mobilebert.png'

    @property
    def label_names(self) -> List[str]:
        # Return keys sorted by value (0, 1, ...)
        return [k for k, v in sorted(self.label_mapping.items(), key=lambda item: item[1])]

# Set random seeds for reproducibility
def set_seed(seed=42):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

# Initialize config and seed
config = Config()
set_seed(config.seed)

# Configure Logging
log_level = logging.INFO if config.verbose else logging.WARNING
logging.basicConfig(
    format="%(asctime)s - %(levelname)s - %(name)s - %(message)s",
    datefmt="%m/%d/%Y %H:%M:%S",
    handlers=[logging.StreamHandler(sys.stdout)],
    level=log_level,
    force=True
)
logger = logging.getLogger("MobileBERT_NSFW")
logger.setLevel(log_level)

In [4]:
!pip install -q transformers accelerate datasets

In [5]:
# Custom Dataset Class
class NSFWDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=128):
        self.texts = texts  # Store original texts for inference/debugging
        self.labels = labels

        # Pre-tokenize all texts efficiently in a batch
        self.encodings = tokenizer(
            [str(t) for t in texts],
            add_special_tokens=True,
            max_length=max_length,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_token_type_ids=True,
            return_tensors='pt'
        )

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            'input_ids': self.encodings['input_ids'][idx],
            'attention_mask': self.encodings['attention_mask'][idx],
            'token_type_ids': self.encodings['token_type_ids'][idx],
            'label': torch.tensor(self.labels[idx], dtype=torch.long)
        }

In [6]:
class Trainer:
    def __init__(self, model, optimizer, scheduler, device, accumulation_steps=1, patience=3):
        self.model = model
        self.optimizer = optimizer
        self.scheduler = scheduler
        self.device = device
        self.accumulation_steps = accumulation_steps
        self.patience = patience
        self.best_val_loss = float('inf')
        self.history = {
            'train_loss': [], 'val_loss': [],
            'train_acc': [], 'val_acc': [],
            'val_precision': [], 'val_recall': [],
            'val_f1': [], 'val_macro_f1': [], 'val_pr_auc': []
        }
        self.patience_counter = 0

    def train_epoch(self, dataloader) -> float:
        self.model.train()
        total_loss = 0

        self.optimizer.zero_grad()

        progress_bar = tqdm(enumerate(dataloader), desc="Training", total=len(dataloader), ncols=100)
        for step, batch in progress_bar:
            input_ids = batch['input_ids'].to(self.device)
            attention_mask = batch['attention_mask'].to(self.device)
            token_type_ids = batch['token_type_ids'].to(self.device)
            labels = batch['label'].to(self.device)

            outputs = self.model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                token_type_ids=token_type_ids,
                labels=labels
            )

            # Normalize loss for gradient accumulation
            loss = outputs.loss / self.accumulation_steps
            total_loss += loss.item() * self.accumulation_steps

            loss.backward()

            if (step + 1) % self.accumulation_steps == 0:
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), 1.0)
                self.optimizer.step()
                self.scheduler.step()
                self.optimizer.zero_grad()

            progress_bar.set_postfix({'loss': f'{loss.item() * self.accumulation_steps:.4f}'})

        avg_loss = total_loss / len(dataloader)
        return avg_loss

    def evaluate(self, dataloader) -> Tuple[float, List, List, List]:
        self.model.eval()
        total_loss = 0
        predictions, true_labels, probabilities = [], [], []

        with torch.no_grad():
            progress_bar = tqdm(dataloader, desc="Evaluating", ncols=100)
            for batch in progress_bar:
                input_ids = batch['input_ids'].to(self.device)
                attention_mask = batch['attention_mask'].to(self.device)
                token_type_ids = batch['token_type_ids'].to(self.device)
                labels = batch['label'].to(self.device)

                outputs = self.model(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    token_type_ids=token_type_ids,
                    labels=labels
                )

                loss = outputs.loss
                total_loss += loss.item()

                # Get probabilities
                probs = torch.softmax(outputs.logits, dim=1)
                probabilities.extend(probs.cpu().numpy())

                preds = torch.argmax(outputs.logits, dim=1)
                predictions.extend(preds.cpu().numpy())
                true_labels.extend(labels.cpu().numpy())

        avg_loss = total_loss / len(dataloader)
        return avg_loss, predictions, true_labels, probabilities

    def train(self, train_loader, val_loader, epochs, tokenizer, save_path='best_model.pth', tokenizer_save_dir='best_mobilebert_tokenizer'):
        logger.info(f"Starting training for {epochs} epochs (Patience={self.patience}, Accumulation={self.accumulation_steps})...")

        for epoch in range(epochs):
            logger.info(f"{'─'*25} EPOCH {epoch + 1}/{epochs} {'─'*25}")

            train_loss = self.train_epoch(train_loader)

            # Calculate training accuracy in a separate pass
            _, train_preds, train_labels, _ = self.evaluate(train_loader)
            train_acc = accuracy_score(train_labels, train_preds)

            val_loss, val_preds, val_labels, val_probs = self.evaluate(val_loader)

            val_acc = accuracy_score(val_labels, val_preds)
            val_precision = precision_score(val_labels, val_preds, average='binary', zero_division=0)
            val_recall = recall_score(val_labels, val_preds, average='binary', zero_division=0)
            val_f1 = f1_score(val_labels, val_preds, average='binary', zero_division=0)
            val_macro_f1 = f1_score(val_labels, val_preds, average='macro', zero_division=0)

            # PR-AUC
            val_probs_np = np.array(val_probs)
            if len(val_probs_np.shape) == 2 and val_probs_np.shape[1] == 2:
                 val_pr_auc = average_precision_score(val_labels, val_probs_np[:, 1])
            else:
                 val_pr_auc = 0.0

            self.history['train_loss'].append(train_loss)
            self.history['val_loss'].append(val_loss)
            self.history['train_acc'].append(train_acc)
            self.history['val_acc'].append(val_acc)
            self.history['val_precision'].append(val_precision)
            self.history['val_recall'].append(val_recall)
            self.history['val_f1'].append(val_f1)
            self.history['val_macro_f1'].append(val_macro_f1)
            self.history['val_pr_auc'].append(val_pr_auc)

            logger.info(f"  📊 Training:   Loss={train_loss:.4f} | Acc={train_acc*100:.2f}%")
            logger.info(f"  📊 Validation: Loss={val_loss:.4f} | Acc={val_acc*100:.2f}%")
            logger.info(f"  📊 Val Metrics: P={val_precision:.4f} | R={val_recall:.4f} | F1={val_f1:.4f} | Macro-F1={val_macro_f1:.4f} | PR-AUC={val_pr_auc:.4f}")

            # Save best model and Early Stopping check
            if val_loss < self.best_val_loss:
                self.best_val_loss = val_loss
                self.patience_counter = 0
                torch.save({
                    'epoch': epoch,
                    'model_state_dict': self.model.state_dict(),
                    'optimizer_state_dict': self.optimizer.state_dict(),
                    'val_loss': val_loss,
                    'val_acc': val_acc,
                    'history': self.history
                }, save_path)

                tokenizer.save_pretrained(tokenizer_save_dir)
                logger.info(f"  ✓ New best model saved! 🎯 (Loss: {val_loss:.4f})")
            else:
                self.patience_counter += 1
                logger.info(f"  ⏳ No improvement. Patience: {self.patience_counter}/{self.patience}")
                if self.patience_counter >= self.patience:
                    logger.info("  🛑 Early stopping triggered!")
                    break

        # Save logs to CSV
        try:
            history_df = pd.DataFrame(self.history)
            history_df.insert(0, 'epoch', range(1, len(history_df) + 1))

            # Save complete training logs
            history_df.to_csv('training_logs.csv', index=False)

            # Save validation specific logs
            val_cols = ['epoch', 'val_loss', 'val_acc', 'val_precision', 'val_recall', 'val_f1', 'val_macro_f1', 'val_pr_auc']
            history_df[val_cols].to_csv('validation_logs.csv', index=False)

            logger.info("  ✓ Logs saved: training_logs.csv, validation_logs.csv")
        except Exception as e:
            logger.warning(f"  ⚠️ Failed to save logs: {e}")

        return self.history

In [7]:
class DataModule:
    def __init__(self, config: Config, tokenizer):
        self.config = config
        self.tokenizer = tokenizer
        self.train_loader = None
        self.val_loader = None
        self.test_loader = None

    def prepare_data(self):
        logger.info(f"Loading dataset from {self.config.csv_path}...")
        if not os.path.exists(self.config.csv_path):
            logger.error(f"CSV file not found: {self.config.csv_path}")
            raise FileNotFoundError(f"CSV file not found: {self.config.csv_path}")

        df = pd.read_csv(self.config.csv_path)

        # Schema Validation & Column Mapping
        expected_cols = [self.config.text_col, self.config.label_col]
        missing_cols = [col for col in expected_cols if col not in df.columns]

        if missing_cols:
            logger.warning(f"Missing columns {missing_cols}. Attempting index-based mapping...")
            if len(df.columns) >= 2:
                # Assume first column is text, second is label (backward compatibility)
                rename_map = {df.columns[0]: self.config.text_col, df.columns[1]: self.config.label_col}
                df = df.rename(columns=rename_map)
                logger.info(f"Renamed columns: {rename_map}")
            else:
                raise ValueError(f"Dataset must have at least 2 columns for {self.config.text_col} and {self.config.label_col}.")
        else:
             logger.info(f"Schema validated: Found columns '{self.config.text_col}' and '{self.config.label_col}'.")

        # Preprocess labels using explicit column names
        if df[self.config.label_col].dtype == object:
            label_map = self.config.label_mapping
            df[self.config.label_col] = df[self.config.label_col].astype(str).str.lower().str.strip().map(label_map)
            df = df.dropna(subset=[self.config.label_col])
            df[self.config.label_col] = df[self.config.label_col].astype(int)

        # Validate allowed labels
        valid_labels = set(self.config.label_mapping.values())
        unique_labels = set(df[self.config.label_col].unique())
        if not unique_labels.issubset(valid_labels):
             invalid = unique_labels - valid_labels
             logger.warning(f"Found invalid labels: {invalid}. Filtering them out...")
             df = df[df[self.config.label_col].isin(valid_labels)]

        logger.info(f"Label validation passed. Unique labels: {sorted(df[self.config.label_col].unique())}")

        texts = df[self.config.text_col].values
        labels = df[self.config.label_col].values

        # Split Logic (70/25/5)
        # 1. First split: Train (70%) vs Rest (30%)
        first_split_size = self.config.val_ratio + self.config.test_ratio

        X_train, X_temp, y_train, y_temp = train_test_split(
            texts, labels,
            test_size=first_split_size,
            random_state=self.config.seed,
            stratify=labels
        )

        # 2. Second split: Rest (30%) into Val (25%) and Test (5%)
        # Proportion of Test inside Rest: 0.05 / 0.30 = 1/6
        second_split_size = self.config.test_ratio / first_split_size

        X_val, X_test, y_val, y_test = train_test_split(
            X_temp, y_temp,
            test_size=second_split_size,
            random_state=self.config.seed,
            stratify=y_temp
        )

        logger.info(f"Split: Train={len(X_train)} ({len(X_train)/len(texts):.1%}), Val={len(X_val)} ({len(X_val)/len(texts):.1%}), Test={len(X_test)} ({len(X_test)/len(texts):.1%})")

        # Datasets
        train_dataset = NSFWDataset(X_train, y_train, self.tokenizer, self.config.max_length)
        val_dataset = NSFWDataset(X_val, y_val, self.tokenizer, self.config.max_length)
        test_dataset = NSFWDataset(X_test, y_test, self.tokenizer, self.config.max_length)

        # Loaders
        self.train_loader = DataLoader(train_dataset, batch_size=self.config.batch_size, shuffle=True)
        self.val_loader = DataLoader(val_dataset, batch_size=self.config.batch_size)
        self.test_loader = DataLoader(test_dataset, batch_size=self.config.batch_size)

        return self.train_loader, self.val_loader, self.test_loader

In [8]:
class Visualizer:
    @staticmethod
    def plot_confusion_matrix(cm, labels=['SFW', 'NSFW'], save_path='confusion_matrix_mobilebert.png'):
        plt.figure(figsize=(10, 8))

        # Calculate row-normalized percentages
        # Add epsilon to avoid division by zero
        cm_sum = cm.sum(axis=1)[:, np.newaxis]
        cm_perc = cm.astype('float') / (cm_sum + 1e-10) * 100

        # Create annotations: Count + (Row %)
        annot = np.empty_like(cm).astype(str)
        nrows, ncols = cm.shape
        for i in range(nrows):
            for j in range(ncols):
                annot[i, j] = f"{cm[i, j]}\n({cm_perc[i, j]:.1f}%)"

        sns.heatmap(cm, annot=annot, fmt='', cmap='RdYlGn_r',
                    xticklabels=labels, yticklabels=labels,
                    cbar_kws={'label': 'Count'}, linewidths=2, linecolor='white')

        plt.xlabel('Predicted Label', fontsize=13, fontweight='bold')
        plt.ylabel('True Label', fontsize=13, fontweight='bold')
        plt.title('Confusion Matrix', fontsize=15, fontweight='bold', pad=20)
        plt.tight_layout()
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        logger.info(f"Confusion matrix saved to {save_path}")
        plt.close()

    @staticmethod
    def plot_history(history, save_path='training_history_mobilebert.png'):
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
        epochs = range(1, len(history['train_loss']) + 1)

        ax1.plot(epochs, history['train_loss'], 'b-o', label='Training Loss')
        ax1.plot(epochs, history['val_loss'], 'r-o', label='Validation Loss')
        ax1.set_title('Loss')
        ax1.legend()
        ax1.grid(True)

        ax2.plot(epochs, [x*100 for x in history['train_acc']], 'b-o', label='Train Acc')
        ax2.plot(epochs, [x*100 for x in history['val_acc']], 'r-o', label='Val Acc')
        ax2.set_title('Accuracy')
        ax2.legend()
        ax2.grid(True)

        plt.tight_layout()
        plt.savefig(save_path, dpi=300)
        logger.info(f"Training history saved to {save_path}")
        plt.close()

    @staticmethod
    def plot_metrics(accuracy, precision, recall, f1, save_path='metrics_comparison.png'):
        metrics = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
        values = [accuracy * 100, precision * 100, recall * 100, f1 * 100]
        plt.figure(figsize=(10, 6))
        bars = plt.bar(metrics, values, color=['#2ecc71', '#3498db', '#e74c3c', '#f39c12'])
        for bar, val in zip(bars, values):
            plt.text(bar.get_x() + bar.get_width()/2, bar.get_height(), f'{val:.2f}%', ha='center', va='bottom')
        plt.ylim(0, 110)
        plt.title('Performance Metrics')
        plt.savefig(save_path, dpi=300)
        logger.info(f"Metrics saved to {save_path}")
        plt.close()

In [9]:
# Plot Training History
def plot_training_history(train_losses, val_losses, train_accs, val_accs,
                         save_path='training_history_mobilebert.png'):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
    epochs = range(1, len(train_losses) + 1)

    # Loss plot
    ax1.plot(epochs, train_losses, 'b-o', label='Training Loss',
             linewidth=2.5, markersize=8)
    ax1.plot(epochs, val_losses, 'r-o', label='Validation Loss',
             linewidth=2.5, markersize=8)
    ax1.set_xlabel('Epoch', fontsize=13, fontweight='bold')
    ax1.set_ylabel('Loss', fontsize=13, fontweight='bold')
    ax1.set_title('Training and Validation Loss', fontsize=15, fontweight='bold')
    ax1.legend(fontsize=11, loc='best')
    ax1.grid(True, alpha=0.3, linestyle='--')
    ax1.set_xticks(epochs)

    # Accuracy plot
    ax2.plot(epochs, [acc*100 for acc in train_accs], 'b-o',
             label='Training Accuracy', linewidth=2.5, markersize=8)
    ax2.plot(epochs, [acc*100 for acc in val_accs], 'r-o',
             label='Validation Accuracy', linewidth=2.5, markersize=8)
    ax2.set_xlabel('Epoch', fontsize=13, fontweight='bold')
    ax2.set_ylabel('Accuracy (%)', fontsize=13, fontweight='bold')
    ax2.set_title('Training and Validation Accuracy', fontsize=15, fontweight='bold')
    ax2.legend(fontsize=11, loc='best')
    ax2.grid(True, alpha=0.3, linestyle='--')
    ax2.set_xticks(epochs)

    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    print(f"✓ Training history saved to {save_path}")
    plt.close()

In [10]:
# Plot Metrics Comparison
def plot_metrics_bar(accuracy, precision, recall, f1,
                     save_path='metrics_comparison_mobilebert.png'):
    metrics = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
    values = [accuracy * 100, precision * 100, recall * 100, f1 * 100]
    colors = ['#2ecc71', '#3498db', '#e74c3c', '#f39c12']

    plt.figure(figsize=(12, 7))
    bars = plt.bar(metrics, values, color=colors, edgecolor='black', linewidth=2)

    # Add value labels on bars
    for bar, value in zip(bars, values):
        height = bar.get_height()
        plt.text(bar.get_x() + bar.get_width()/2., height,
                f'{value:.2f}%',
                ha='center', va='bottom', fontsize=14, fontweight='bold')

    plt.ylabel('Score (%)', fontsize=13, fontweight='bold')
    plt.title('Performance Metrics - MobileBERT NSFW Classifier',
              fontsize=15, fontweight='bold', pad=20)
    plt.ylim(0, 105)
    plt.grid(axis='y', alpha=0.3, linestyle='--')
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    print(f"✓ Metrics comparison saved to {save_path}")
    plt.close()

In [11]:
class Predictor:
    def __init__(self, model, tokenizer, device, max_length=128, threshold=0.5):
        self.model = model
        self.tokenizer = tokenizer
        self.device = device
        self.max_length = max_length
        self.threshold = threshold
        self.model.eval()

    def predict(self, text):
        encoding = self.tokenizer.encode_plus(
            text,
            add_special_tokens=True,
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )

        input_ids = encoding['input_ids'].to(self.device)
        attention_mask = encoding['attention_mask'].to(self.device)
        token_type_ids = encoding['token_type_ids'].to(self.device)

        start_time = time.time()
        with torch.no_grad():
            outputs = self.model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                token_type_ids=token_type_ids
            )
            probs = torch.softmax(outputs.logits, dim=1)

            # Apply Threshold
            if probs.shape[1] == 2:
                # Binary classification: Check if positive class prob >= threshold
                pred_idx = 1 if probs[0, 1] >= self.threshold else 0
            else:
                # Multiclass fallback
                pred_idx = torch.argmax(probs, dim=1).item()

        inference_time = (time.time() - start_time) * 1000
        return pred_idx, probs.cpu().numpy()[0], inference_time

In [12]:
class ModelExporter:
    @staticmethod
    def export_onnx(model, tokenizer, onnx_path="mobilebert_nsfw.onnx", max_length=128):
        logger.info(f"Exporting to ONNX: {onnx_path}...")

        # Define wrapper to explicitly map inputs
        class MobileBertWrapper(nn.Module):
            def __init__(self, model):
                super().__init__()
                self.model = model

            def forward(self, input_ids, attention_mask, token_type_ids):
                return self.model(input_ids=input_ids, attention_mask=attention_mask, token_type_ids=token_type_ids).logits

        wrapped_model = MobileBertWrapper(model)
        wrapped_model.eval()
        device = next(model.parameters()).device

        dummy_input = tokenizer("Dummy text", return_tensors="pt", max_length=max_length, padding="max_length", truncation=True)
        dummy_input = {k: v.to(device) for k, v in dummy_input.items()}

        try:
            torch.onnx.export(
                wrapped_model,
                (dummy_input["input_ids"], dummy_input["attention_mask"], dummy_input["token_type_ids"]),
                onnx_path,
                export_params=True,
                opset_version=14,
                do_constant_folding=True,
                input_names=['input_ids', 'attention_mask', 'token_type_ids'],
                output_names=['logits'],
                dynamic_axes={'input_ids': {0: 'batch'}, 'logits': {0: 'batch'}}
            )
            logger.info(f"  \u2713 ONNX export success! Size: {os.path.getsize(onnx_path)/(1024**2):.2f} MB")
            return True
        except Exception as e:
            logger.error(f"  \u274c ONNX export failed: {e}")
            return False

    @staticmethod
    def export_tflite(pt_model, model_name, output_dir=".", max_len=128):
        """Convert the trained MobileBERT to TFLite via litert-torch (LiteRT).

        transformers v5 dropped native TF, so the old TFAutoModel(from_pt=True) ->
        TFLite route no longer works (it also produced a bloated ~48 MB FP16 file).
        litert-torch (formerly 'ai-edge-torch') converts PyTorch -> LiteRT directly
        and supports a data-free dynamic-range INT8 recipe. This matches the
        DistilBERT / DOST_RoBERTa / TinyBERT export path, so every transformer in
        the study ships comparable artifacts and we can report a fair on-device size.

        MobileBERT needs three inputs (input_ids, attention_mask, token_type_ids),
        and MobileBertForSequenceClassification returns a SequenceClassifierOutput,
        so a thin wrapper exposes the logits tensor that torch.export can trace.

        Emits:
          - mobilebert_nsfw_float32.tflite : reference build (largest)
          - mobilebert_nsfw_int8.tflite    : dynamic-range INT8 weights (~1/4 size)
        Both import via Android Studio's ML Model Binding.
        """
        import importlib
        logger.info("Exporting MobileBERT to TFLite (litert-torch)...")

        # 'ai-edge-torch' was renamed to 'litert-torch'; the old name is now a stub
        # without .convert. Use whichever module actually provides the API.
        aet = None
        for modname in ('litert_torch', 'ai_edge_torch'):
            try:
                m = importlib.import_module(modname)
            except Exception:
                continue
            if hasattr(m, 'convert'):
                aet = m
                break
        if aet is None:
            logger.error("No working litert_torch/ai_edge_torch (.convert) found; skipping TFLite export.")
            return False
        pkg = aet.__name__
        logger.info(f"  Using '{pkg}' for TFLite conversion.")

        # Thin wrapper: 3 positional inputs in, logits tensor out (torch.export-friendly).
        class MobileBertLogits(nn.Module):
            def __init__(self, model):
                super().__init__()
                self.model = model

            def forward(self, input_ids, attention_mask, token_type_ids):
                return self.model(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    token_type_ids=token_type_ids,
                ).logits

        try:
            # Rebuild a clean CPU eval copy with eager attention. eager avoids
            # data-dependent attention branches that torch.export cannot trace.
            try:
                base = MobileBertForSequenceClassification.from_pretrained(
                    model_name, num_labels=2, attn_implementation="eager"
                )
            except Exception:
                base = MobileBertForSequenceClassification.from_pretrained(
                    model_name, num_labels=2
                )
            base.load_state_dict(pt_model.state_dict())
            export_model = MobileBertLogits(base).eval().to("cpu")

            tokenizer = MobileBertTokenizer.from_pretrained(model_name)
            enc = tokenizer(
                "Ito ay isang sample na Tagalog text", return_tensors="pt",
                max_length=max_len, padding="max_length", truncation=True
            )
            sample = (enc["input_ids"], enc["attention_mask"], enc["token_type_ids"])

            os.makedirs(output_dir, exist_ok=True)
            ok = False

            # --- Float32 LiteRT model (reliable baseline) ---
            try:
                logger.info(f"  Converting PyTorch -> TFLite (float32) via {pkg}...")
                edge = aet.convert(export_model, sample)
                f32 = os.path.join(output_dir, "mobilebert_nsfw_float32.tflite")
                edge.export(f32)
                logger.info(f"  \u2713 TFLite (float32) saved to {f32} ({os.path.getsize(f32)/(1024**2):.2f} MB)")
                ok = True
            except Exception as e:
                logger.error(f"  {pkg} float32 conversion failed: {e}")
                import traceback
                traceback.print_exc()

            # --- Dynamic-range INT8 (data-free; ~1/4 size) ---
            try:
                pt2e = importlib.import_module(f"{pkg}.quantize.pt2e_quantizer")
                qcfg = importlib.import_module(f"{pkg}.quantize.quant_config")
                PT2EQuantizer = pt2e.PT2EQuantizer
                get_symmetric_quantization_config = pt2e.get_symmetric_quantization_config
                QuantConfig = qcfg.QuantConfig

                # prepare_pt2e/convert_pt2e moved out of torch.ao into torchao in newer torch.
                prepare_pt2e = convert_pt2e = None
                for modpath in (
                    'torchao.quantization.pt2e.quantize_pt2e',
                    'torch.ao.quantization.quantize_pt2e',
                    'torch.ao.quantization.pt2e.quantize_pt2e',
                ):
                    try:
                        qmod = importlib.import_module(modpath)
                        prepare_pt2e, convert_pt2e = qmod.prepare_pt2e, qmod.convert_pt2e
                        break
                    except Exception:
                        continue
                if prepare_pt2e is None:
                    raise ImportError("prepare_pt2e/convert_pt2e not found (torch.ao or torchao)")

                logger.info(f"  Converting PyTorch -> TFLite (dynamic-range INT8) via {pkg} PT2E...")
                quantizer = PT2EQuantizer().set_global(
                    get_symmetric_quantization_config(is_per_channel=True, is_dynamic=True)
                )

                # Capture a pre-autograd graph (entry point name varies across torch versions).
                captured = None
                try:
                    captured = torch.export.export_for_training(export_model, sample).module()
                except (AttributeError, TypeError):
                    try:
                        from torch._export import capture_pre_autograd_graph
                        captured = capture_pre_autograd_graph(export_model, sample)
                    except Exception:
                        captured = torch.export.export(export_model, sample).module()

                prepared = prepare_pt2e(captured, quantizer)
                converted = convert_pt2e(prepared)
                edge_q = aet.convert(
                    converted, sample, quant_config=QuantConfig(pt2e_quantizer=quantizer)
                )
                i8 = os.path.join(output_dir, "mobilebert_nsfw_int8.tflite")
                edge_q.export(i8)
                logger.info(f"  \u2713 TFLite (INT8) saved to {i8} ({os.path.getsize(i8)/(1024**2):.2f} MB)")
                ok = True
            except Exception as e:
                logger.error(f"  {pkg} INT8 conversion failed: {e}")
                import traceback
                traceback.print_exc()

            if ok:
                logger.info("  Import in Android Studio via: File > New > Other > TensorFlow Lite Model.")
            else:
                logger.error(f"  {pkg} produced no TFLite files.")
            return ok

        except Exception as e:
            logger.error(f"  \u274c TFLite export failed: {e}")
            import traceback
            traceback.print_exc()
            return False


In [13]:
# 1. SETUP & INITIALIZATION
logger.info("="*50 + "\n🚀 STEP 1: SETUP & INITIALIZATION\n" + "="*50)

# Initialize Model & Tokenizer
logger.info("Initializing Model and Tokenizer...")
tokenizer = MobileBertTokenizer.from_pretrained(config.model_name)
model = MobileBertForSequenceClassification.from_pretrained(
    config.model_name,
    num_labels=2,
    hidden_dropout_prob=config.dropout,
    attention_probs_dropout_prob=config.dropout,
)
model.to(config.device)

# Model Summary
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
model_size_mb = model.get_memory_footprint() / (1024**2)

logger.info(f"✓ MobileBERT initialized!")
logger.info(f"  ├─ Model: {config.model_name}")
logger.info(f"  ├─ Device: {config.device}")
logger.info(f"  ├─ Parameters: {total_params:,}")
logger.info(f"  ├─ Trainable: {trainable_params:,}")
logger.info(f"  └─ Model Size: ~{model_size_mb:.2f} MB")

# Prepare Data
logger.info("Preparing Data...")
data_module = DataModule(config, tokenizer)
train_loader, val_loader, test_loader = data_module.prepare_data()

# Setup Training
logger.info("Configuring Optimizer & Trainer...")
optimizer = AdamW(model.parameters(), lr=config.learning_rate, weight_decay=config.weight_decay)

# Adjust total steps for gradient accumulation
num_update_steps_per_epoch = len(train_loader) // config.gradient_accumulation_steps
total_steps = num_update_steps_per_epoch * config.epochs

scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=int(config.warmup_ratio*total_steps), num_training_steps=total_steps)

trainer = Trainer(model, optimizer, scheduler, config.device,
                 accumulation_steps=config.gradient_accumulation_steps,
                 patience=config.patience)
logger.info("✓ Setup Complete. Proceed to training.")

06/30/2026 08:08:59 - INFO - MobileBERT_NSFW - ==================================================
🚀 STEP 1: SETUP & INITIALIZATION
06/30/2026 08:08:59 - INFO - MobileBERT_NSFW - Initializing Model and Tokenizer...


vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/847 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/147M [00:00<?, ?B/s]

Some weights of MobileBertForSequenceClassification were not initialized from the model checkpoint at google/mobilebert-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


model.safetensors:   0%|          | 0.00/147M [00:00<?, ?B/s]

06/30/2026 08:09:13 - INFO - MobileBERT_NSFW - ✓ MobileBERT initialized!
06/30/2026 08:09:13 - INFO - MobileBERT_NSFW -   ├─ Model: google/mobilebert-uncased
06/30/2026 08:09:13 - INFO - MobileBERT_NSFW -   ├─ Device: cuda
06/30/2026 08:09:13 - INFO - MobileBERT_NSFW -   ├─ Parameters: 24,582,914
06/30/2026 08:09:13 - INFO - MobileBERT_NSFW -   ├─ Trainable: 24,582,914
06/30/2026 08:09:13 - INFO - MobileBERT_NSFW -   └─ Model Size: ~93.78 MB
06/30/2026 08:09:13 - INFO - MobileBERT_NSFW - Preparing Data...
06/30/2026 08:09:13 - INFO - MobileBERT_NSFW - Loading dataset from cleaned_data.csv...
06/30/2026 08:09:13 - INFO - MobileBERT_NSFW - Schema validated: Found columns 'text' and 'label'.
06/30/2026 08:09:13 - INFO - MobileBERT_NSFW - Label validation passed. Unique labels: [np.int64(0), np.int64(1)]
06/30/2026 08:09:13 - INFO - MobileBERT_NSFW - Split: Train=3905 (70.0%), Val=1394 (25.0%), Test=280 (5.0%)
06/30/2026 08:09:15 - INFO - MobileBERT_NSFW - Configuring Optimizer & Trainer..

In [14]:
# 2. TRAINING
logger.info("="*50 + "\n🚀 STEP 2: TRAINING MODEL\n" + "="*50)

history = trainer.train(train_loader, val_loader, config.epochs, tokenizer,
                       save_path=config.model_save_name,
                       tokenizer_save_dir=config.tokenizer_save_dir)
logger.info("✓ Training finished. Best model saved.")

06/30/2026 08:09:15 - INFO - MobileBERT_NSFW - ==================================================
🚀 STEP 2: TRAINING MODEL
06/30/2026 08:09:15 - INFO - MobileBERT_NSFW - Starting training for 50 epochs (Patience=3, Accumulation=1)...
06/30/2026 08:09:15 - INFO - MobileBERT_NSFW - ───────────────────────── EPOCH 1/50 ─────────────────────────



Evaluating: 100%|███████████████████████████████████████████████████| 44/44 [00:01<00:00, 26.04it/s]


06/30/2026 08:09:44 - INFO - MobileBERT_NSFW -   📊 Training:   Loss=2118084.0935 | Acc=55.19%
06/30/2026 08:09:44 - INFO - MobileBERT_NSFW -   📊 Validation: Loss=548534.0419 | Acc=55.16%
06/30/2026 08:09:44 - INFO - MobileBERT_NSFW -   📊 Val Metrics: P=0.0000 | R=0.0000 | F1=0.0000 | Macro-F1=0.3555 | PR-AUC=0.4484
06/30/2026 08:09:45 - INFO - MobileBERT_NSFW -   ✓ New best model saved! 🎯 (Loss: 548534.0419)
06/30/2026 08:09:45 - INFO - MobileBERT_NSFW - ───────────────────────── EPOCH 2/50 ─────────────────────────


Evaluating: 100%|███████████████████████████████████████████████████| 44/44 [00:01<00:00, 26.74it/s]


06/30/2026 08:10:13 - INFO - MobileBERT_NSFW -   📊 Training:   Loss=274052.4409 | Acc=54.96%
06/30/2026 08:10:13 - INFO - MobileBERT_NSFW -   📊 Validation: Loss=147.2499 | Acc=54.95%
06/30/2026 08:10:13 - INFO - MobileBERT_NSFW -   📊 Val Metrics: P=0.0000 | R=0.0000 | F1=0.0000 | Macro-F1=0.3546 | PR-AUC=0.3859
06/30/2026 08:10:14 - INFO - MobileBERT_NSFW -   ✓ New best model saved! 🎯 (Loss: 147.2499)
06/30/2026 08:10:14 - INFO - MobileBERT_NSFW - ───────────────────────── EPOCH 3/50 ─────────────────────────


...

06/30/2026 08:14:28 - INFO - MobileBERT_NSFW -   📊 Training:   Loss=3.4967 | Acc=91.81%
06/30/2026 08:14:28 - INFO - MobileBERT_NSFW -   📊 Validation: Loss=0.4327 | Acc=86.23%
06/30/2026 08:14:28 - INFO - MobileBERT_NSFW -   📊 Val Metrics: P=0.8357 | R=0.8624 | F1=0.8488 | Macro-F1=0.8612 | PR-AUC=0.9360
06/30/2026 08:14:28 - INFO - MobileBERT_NSFW -   ⏳ No improvement. Patience: 1/3
06/30/2026 08:14:28 - INFO - MobileBERT_NSFW - ───────────────────────── EPOCH 12/50 ─────────────────────────


Evaluating: 100%|███████████████████████████████████████████████████| 44/44 [00:01<00:00, 26.86it/s]


06/30/2026 08:14:56 - INFO - MobileBERT_NSFW -   📊 Training:   Loss=1.1307 | Acc=92.45%
06/30/2026 08:14:56 - INFO - MobileBERT_NSFW -   📊 Validation: Loss=0.4193 | Acc=87.52%
06/30/2026 08:14:56 - INFO - MobileBERT_NSFW -   📊 Val Metrics: P=0.8475 | R=0.8800 | F1=0.8634 | Macro-F1=0.8742 | PR-AUC=0.9388
06/30/2026 08:14:56 - INFO - MobileBERT_NSFW -   ⏳ No improvement. Patience: 2/3
06/30/2026 08:14:56 - INFO - MobileBERT_NSFW - ───────────────────────── EPOCH 13/50 ─────────────────────────


Evaluating: 100%|███████████████████████████████████████████████████| 44/44 [00:01<00:00, 26.16it/s]


06/30/2026 08:15:23 - INFO - MobileBERT_NSFW -   📊 Training:   Loss=7.0519 | Acc=93.47%
06/30/2026 08:15:23 - INFO - MobileBERT_NSFW -   📊 Validation: Loss=0.3843 | Acc=88.02%
06/30/2026 08:15:23 - INFO - MobileBERT_NSFW -   📊 Val Metrics: P=0.8623 | R=0.8720 | F1=0.8671 | Macro-F1=0.8790 | PR-AUC=0.9433
06/30/2026 08:15:23 - INFO - MobileBERT_NSFW -   ⏳ No improvement. Patience: 3/3
06/30/2026 08:15:23 - INFO - MobileBERT_NSFW -   🛑 Early stopping triggered!
06/30/2026 08:15:23 - INFO - MobileBERT_NSFW -   ✓ Logs saved: training_logs.csv, validation_logs.csv
06/30/2026 08:15:23 - INFO - MobileBERT_NSFW - ✓ Training finished. Best model saved.


In [15]:
# 3. EVALUATION & VISUALIZATION
logger.info("="*50 + "\n🚀 STEP 3: EVALUATION & VISUALIZATION\n" + "="*50)

# Plot History
Visualizer.plot_history(history, save_path=config.plot_history_path)

# Evaluate on Test Set
logger.info("Evaluating on Test Set...")
# Load best model (Frozen state)
checkpoint = torch.load(config.model_save_name, weights_only=False)
model.load_state_dict(checkpoint['model_state_dict'])

# --- NEW: Tune Threshold on Validation Set ---
logger.info("Tuning Threshold on Validation Set to maximize F1-score...")
_, _, val_labels, val_probs = trainer.evaluate(val_loader)
val_probs_positive = np.array(val_probs)[:, 1]

best_threshold = 0.5
best_val_f1 = 0.0
thresholds = np.arange(0.01, 1.00, 0.01)

for thresh in thresholds:
    thresh_preds = (val_probs_positive >= thresh).astype(int)
    score = f1_score(val_labels, thresh_preds, average='binary', zero_division=0)
    if score > best_val_f1:
        best_val_f1 = score
        best_threshold = thresh

logger.info(f"  🎯 Optimal Threshold found: {best_threshold:.2f} (Val F1: {best_val_f1:.4f})")
config.best_threshold = best_threshold # Store for inference

# Generate and Save Validation Report
val_predictions = (val_probs_positive >= best_threshold).astype(int)
val_report = classification_report(val_labels, val_predictions, target_names=config.label_names)
with open("validation_classification_report.txt", "w") as f:
    f.write(val_report)
logger.info("  ✓ Validation classification report saved to validation_classification_report.txt")
# ---------------------------------------------

# Evaluate on Test Set using Optimal Threshold
logger.info(f"Evaluating on Test Set with Threshold={best_threshold:.2f}...")
avg_loss, _, true_labels, probabilities = trainer.evaluate(test_loader)

probs_np = np.array(probabilities)
# Use tuned threshold for predictions
if len(probs_np.shape) == 2 and probs_np.shape[1] == 2:
    positive_probs = probs_np[:, 1]
    predictions = (positive_probs >= best_threshold).astype(int)
    pr_auc = average_precision_score(true_labels, positive_probs)
else:
    predictions = np.argmax(probs_np, axis=1)
    pr_auc = 0.0

# Metrics
acc = accuracy_score(true_labels, predictions)
prec = precision_score(true_labels, predictions, average='binary', zero_division=0)
rec = recall_score(true_labels, predictions, average='binary', zero_division=0)
f1 = f1_score(true_labels, predictions, average='binary', zero_division=0)
macro_f1 = f1_score(true_labels, predictions, average='macro', zero_division=0)

cm = confusion_matrix(true_labels, predictions)

logger.info(f"  🏆 Test Results: Acc={acc:.4f}, P={prec:.4f}, R={rec:.4f}, F1={f1:.4f}, Macro-F1={macro_f1:.4f}, PR-AUC={pr_auc:.4f}")
test_report = classification_report(true_labels, predictions, target_names=config.label_names)
logger.info("\n" + test_report)

# Save Test Report
with open("test_classification_report.txt", "w") as f:
    f.write(test_report)
logger.info("  ✓ Test classification report saved to test_classification_report.txt")

Visualizer.plot_confusion_matrix(cm, labels=config.label_names, save_path=config.plot_cm_path)
Visualizer.plot_metrics(acc, prec, rec, f1, save_path=config.plot_metrics_path)

06/30/2026 08:15:23 - INFO - MobileBERT_NSFW - ==================================================
🚀 STEP 3: EVALUATION & VISUALIZATION
06/30/2026 08:15:24 - INFO - MobileBERT_NSFW - Training history saved to training_history_mobilebert.png
06/30/2026 08:15:24 - INFO - MobileBERT_NSFW - Evaluating on Test Set...
06/30/2026 08:15:25 - INFO - MobileBERT_NSFW - Tuning Threshold on Validation Set to maximize F1-score...


Evaluating: 100%|███████████████████████████████████████████████████| 44/44 [00:01<00:00, 26.76it/s]


06/30/2026 08:15:27 - INFO - MobileBERT_NSFW -   🎯 Optimal Threshold found: 0.51 (Val F1: 0.8576)
06/30/2026 08:15:27 - INFO - MobileBERT_NSFW -   ✓ Validation classification report saved to validation_classification_report.txt
06/30/2026 08:15:27 - INFO - MobileBERT_NSFW - Evaluating on Test Set with Threshold=0.51...


Evaluating: 100%|█████████████████████████████████████████████████████| 9/9 [00:00<00:00, 26.69it/s]

06/30/2026 08:15:27 - INFO - MobileBERT_NSFW -   🏆 Test Results: Acc=0.8893, P=0.9123, R=0.8320, F1=0.8703, Macro-F1=0.8869, PR-AUC=0.9509
06/30/2026 08:15:27 - INFO - MobileBERT_NSFW - 
              precision    recall  f1-score   support

        safe       0.87      0.94      0.90       155
        nsfw       0.91      0.83      0.87       125

    accuracy                           0.89       280
   macro avg       0.89      0.88      0.89       280
weighted avg       0.89      0.89      0.89       280

06/30/2026 08:15:27 - INFO - MobileBERT_NSFW -   ✓ Test classification report saved to test_classification_report.txt


06/30/2026 08:15:27 - INFO - MobileBERT_NSFW - Confusion matrix saved to confusion_matrix_mobilebert.png
06/30/2026 08:15:28 - INFO - MobileBERT_NSFW - Metrics saved to metrics_comparison_mobilebert.png


In [16]:
# 4. INFERENCE & EXPORT
logger.info("="*50 + "\n🚀 STEP 4: INFERENCE & EXPORT\n" + "="*50)

# 4.1 Inference Examples
logger.info("Running Inference Examples (Sampled from Test Set)...")

# Use the tuned threshold if available, else 0.5
threshold = getattr(config, 'best_threshold', 0.5)
logger.info(f"Initializing Predictor with threshold: {threshold:.2f}")

predictor = Predictor(model, tokenizer, config.device, threshold=threshold)
inference_logs = []

# Sample from test dataset
if hasattr(test_loader.dataset, 'texts'):
    import random
    dataset_len = len(test_loader.dataset)
    # Sample 5 random indices
    indices = random.sample(range(dataset_len), min(5, dataset_len))

    logger.info(f"  🔍 Inspecting {len(indices)} random samples from Test Set:")

    for idx in indices:
        text = str(test_loader.dataset.texts[idx])
        true_label_id = test_loader.dataset.labels[idx]
        true_label = config.label_names[true_label_id]

        pred, probs, t = predictor.predict(text)
        pred_label = config.label_names[pred] if pred < len(config.label_names) else f"Label {pred}"
        confidence = probs[pred]

        # Check if prediction matches true label
        status = "✅" if pred == true_label_id else "❌"

        logger.info(f"  {status} Text: \"{text[:60]}...\"")
        logger.info(f"     True: {true_label} | Pred: {pred_label} ({confidence*100:.1f}%) | Time: {t:.2f}ms")

        inference_logs.append({
            "text_snippet": text[:100],
            "true_label": true_label,
            "predicted_label": pred_label,
            "confidence": confidence,
            "inference_time_ms": t,
            "correct": pred == true_label_id
        })
else:
    # Fallback if texts are not available
    logger.warning("Original texts not found in dataset. Using static examples.")
    test_examples = ["Magandang umaga po!", "Kumusta ka na?", "Puta", "Fuck you", "Gutom na ako"]
    for text in test_examples:
        pred, probs, t = predictor.predict(text)
        label = config.label_names[pred] if pred < len(config.label_names) else f"Label {pred}"
        confidence = probs[pred]
        logger.info(f"  📝 '{text}' -> {label} ({confidence*100:.1f}%) [{t:.2f}ms]")

        inference_logs.append({
            "text_snippet": text,
            "true_label": None,
            "predicted_label": label,
            "confidence": confidence,
            "inference_time_ms": t,
            "correct": None
        })

# Save Inference Logs
if inference_logs:
    try:
        pd.DataFrame(inference_logs).to_csv("inference_logs.csv", index=False)
        logger.info("  ✓ Inference logs saved to inference_logs.csv")
    except Exception as e:
        logger.warning(f"  ⚠️ Failed to save inference logs: {e}")

# 4.2 Exports
logger.info("Exporting Models...")
ModelExporter.export_onnx(model, tokenizer, onnx_path=config.onnx_path, max_length=config.max_length)
ModelExporter.export_tflite(model, config.model_name, output_dir=config.output_dir, max_len=config.max_length)

# 4.3 Combine Tokenizer JSON (For Android)
logger.info("Creating Combined Tokenizer JSON...")
tokenizer_dir = config.tokenizer_save_dir
if os.path.exists(tokenizer_dir):
    try:
        with open(os.path.join(tokenizer_dir, 'tokenizer_config.json'), 'r') as f: config_json = json.load(f)
        with open(os.path.join(tokenizer_dir, 'special_tokens_map.json'), 'r') as f: specials = json.load(f)
        vocab = {}
        with open(os.path.join(tokenizer_dir, 'vocab.txt'), 'r') as f:
            for i, line in enumerate(f): vocab[line.strip()] = i

        with open('mobilebert_tokenizer_combined.json', 'w') as f:
            json.dump({"config": config_json, "special_tokens": specials, "vocab": vocab}, f)
        logger.info("  ✓ mobilebert_tokenizer_combined.json created.")
    except Exception as e:
        logger.error(f"  ❌ Error combining tokenizer: {e}")

06/30/2026 08:15:28 - INFO - MobileBERT_NSFW - ==================================================
🚀 STEP 4: INFERENCE & EXPORT
06/30/2026 08:15:28 - INFO - MobileBERT_NSFW - Running Inference Examples (Sampled from Test Set)...
06/30/2026 08:15:28 - INFO - MobileBERT_NSFW - Initializing Predictor with threshold: 0.51
06/30/2026 08:15:28 - INFO - MobileBERT_NSFW -   🔍 Inspecting 5 random samples from Test Set:
06/30/2026 08:15:28 - INFO - MobileBERT_NSFW -   ✅ Text: "kulay rosas ang butas..."
06/30/2026 08:15:28 - INFO - MobileBERT_NSFW -      True: safe | Pred: safe (83.8%) | Time: 53.34ms
06/30/2026 08:15:28 - INFO - MobileBERT_NSFW -   ✅ Text: "when is it my turn to..."
06/30/2026 08:15:28 - INFO - MobileBERT_NSFW -      True: safe | Pred: safe (98.6%) | Time: 36.09ms
06/30/2026 08:15:28 - INFO - MobileBERT_NSFW -   ✅ Text: "nigga you weird asf..."
06/30/2026 08:15:28 - INFO - MobileBERT_NSFW -      True: safe | Pred: safe (89.5%) | Time: 35.76ms
06/30/2026 08:15:28 - INFO - MobileBE

W0630 08:15:29.195000 9144 torch/onnx/_internal/exporter/_compat.py:133] Setting ONNX exporter to use operator set version 18 because the requested opset_version 14 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >=18 to leverage latest ONNX features


[torch.onnx] Obtain model graph for `MobileBertWrapper([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `MobileBertWrapper([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...
[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
06/30/2026 08:15:52 - INFO - onnx_ir.passes.common.unused_removal - No unused functions to remove
06/30/2026 08:15:52 - WARNING - onnxscript.version_converter - The model version conversion is not supported by the onnxscript version converter and fallback is enabled. The model will be converted using the onnx C API (target version: 14).
06/30/2026 08:15:53 - WARNING - onnxscript.version_converter - Failed to convert the model to the target version 14 using the ONNX C API. The model was not modified
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/onnxscript/version_conv

Some weights of MobileBertForSequenceClassification were not initialized from the model checkpoint at google/mobilebert-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


06/30/2026 08:16:02 - INFO - MobileBERT_NSFW -   Converting PyTorch -> TFLite (float32) via litert_torch...


...

(00:59) [START] LiteRT-Torch Convert > Run LiteRT Converter Passes

(01:00) [ FAIL] LiteRT-Torch Convert > Run LiteRT Converter Passes

(01:00) [ FAIL] LiteRT-Torch Convert

06/30/2026 08:18:00 - ERROR - MobileBERT_NSFW -   litert_torch INT8 conversion failed: Failed to run converter passes: /usr/local/lib/python3.12/dist-packages/torch/nn/modules/linear.py:134:0: error: 'stablehlo.uniform_dequantize' op operand #0 must be ranked tensor of per-tensor integer quantized or per-axis integer quantized values, but got 'tensor<512x384xi8>'

06/30/2026 08:18:00 - INFO - MobileBERT_NSFW -   Import in Android Studio via: File > New > Other > TensorFlow Lite Model.
06/30/2026 08:18:00 - INFO - MobileBERT_NSFW - Creating Combined Tokenizer JSON...
06/30/2026 08:18:00 - INFO - MobileBERT_NSFW -   ✓ mobilebert_tokenizer_combined.json created.


Traceback (most recent call last):
  File "/tmp/ipykernel_9144/269099812.py", line 172, in export_tflite
    edge_q = aet.convert(
             ^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/litert_torch/_convert/interface.py", line 323, in convert
    return Converter().convert(
           ^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/litert_torch/_convert/interface.py", line 208, in convert
    converted_model = core.convert_signatures(
                      ^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/contextlib.py", line 81, in inner
    return func(*args, **kwds)
           ^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/litert_torch/_convert/core.py", line 158, in convert_signatures
    exporter = litert_converter.exported_programs_to_flatbuffer(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/litert_torch/_convert/litert_converter.py", line 189, in ex

In [17]:
# Zip the results folder
!zip -r MobileBERT.zip /content/

  adding: content/ (stored 0%)
  adding: content/.config/ (stored 0%)
  adding: content/.config/config_sentinel (stored 0%)
  adding: content/.config/configurations/ (stored 0%)
  adding: content/.config/configurations/config_default (deflated 15%)
  adding: content/.config/gce (stored 0%)
  adding: content/.config/default_configs.db (deflated 98%)
  adding: content/.config/.last_survey_prompt.yaml (stored 0%)
  adding: content/.config/logs/ (stored 0%)
  adding: content/.config/logs/2026.06.04/ (stored 0%)
  adding: content/.config/logs/2026.06.04/13.39.24.274402.log (deflated 58%)
  adding: content/.config/logs/2026.06.04/13.39.36.079406.log (deflated 57%)
  adding: content/.config/logs/2026.06.04/13.38.54.010366.log (deflated 92%)
  adding: content/.config/logs/2026.06.04/13.39.22.644527.log (deflated 87%)
  adding: content/.config/logs/2026.06.04/13.39.37.138439.log (deflated 56%)
  adding: content/.config/logs/2026.06.04/13.39.12.467650.log (deflated 58%)
  adding: content/.config

In [18]:
from google.colab import files

files.download('MobileBERT.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>